In [1]:
# imports
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy import linalg, signal
from scipy.signal import find_peaks

mne.set_log_level("WARNING")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True


In [2]:
# paths
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "outputs" / "figures"
qc_dir = project_root / "outputs" / "qc"

for folder in [data_raw_dir, data_processed_dir, figures_dir, qc_dir]:
    folder.mkdir(parents=True, exist_ok=True)

fif_path = data_raw_dir / "raw_artefacts_emg.fif"
annotations_path = qc_dir / "raw_base-annot.fif"
legacy_annotations_path = qc_dir / "raw_base_annotations.csv"

# Альтернативы для ручного запуска:
# fif_path = Path(r"/Users/user/Desktop/raw_artefacts_emg.fif")
# fif_path = Path(r"C:\\Users\\user\\Desktop\\raw_artefacts_emg.fif")

print("Project root:", project_root)
print("Input FIF:", fif_path)
print("Processed:", data_processed_dir)
print("Figures:", figures_dir)
print("QC:", qc_dir)


Project root: /Users/user/emg_artifacts_filter
Input FIF: /Users/user/emg_artifacts_filter/data/raw/raw_artefacts_emg.fif
Processed: /Users/user/emg_artifacts_filter/data/processed
Figures: /Users/user/emg_artifacts_filter/outputs/figures
QC: /Users/user/emg_artifacts_filter/outputs/qc


In [3]:
# clone raw
if not fif_path.exists():
    raise FileNotFoundError(f"Файл не найден: {fif_path}")

raw_original = mne.io.read_raw_fif(fif_path, preload=True)
raw_base = raw_original.copy()

# При повторном запуске восстанавливаем сохранённую разметку.
if annotations_path.exists():
    saved_annotations = mne.read_annotations(annotations_path)
    raw_base.set_annotations(saved_annotations)
    annotations_source = annotations_path
elif legacy_annotations_path.exists():
    # Однократная совместимость со старым CSV: MNE записал onset как дату от 1970-01-01.
    annotations_table = pd.read_csv(legacy_annotations_path)
    epoch = pd.Timestamp("1970-01-01", tz="UTC")
    annotation_onsets = (
        pd.to_datetime(annotations_table["onset"], utc=True) - epoch
    ).dt.total_seconds().to_numpy()
    saved_annotations = mne.Annotations(
        onset=annotation_onsets,
        duration=annotations_table["duration"].to_numpy(),
        description=annotations_table["description"].astype(str).to_numpy(),
        orig_time=None,
    )
    raw_base.set_annotations(saved_annotations)
    annotations_source = legacy_annotations_path
else:
    annotations_source = None

sfreq = raw_base.info["sfreq"]
duration_s = raw_base.n_times / sfreq

print("File:", fif_path.name)
print("Channels:", len(raw_base.ch_names))
print("sfreq:", sfreq)
print("Duration, s:", round(duration_s, 2))
print("First channels:", raw_base.ch_names[:10])
print("Annotations:", len(raw_base.annotations))
print("Annotations loaded from:", annotations_source or "input FIF")


/var/folders/sw/007c5hsj5zggxfp7nts4rq2w0000gn/T/ipykernel_32210/3011246944.py:5: RuntimeWarning: This filename (/Users/user/emg_artifacts_filter/data/raw/raw_artefacts_emg.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_original = mne.io.read_raw_fif(fif_path, preload=True)


File: raw_artefacts_emg.fif
Channels: 8
sfreq: 4000.0
Duration, s: 117.52
First channels: ['RF R', 'BF R', 'TA R', 'GM R', 'RF L', 'BF L', 'TA L', 'GM L']
Annotations: 19
Annotations loaded from: /Users/user/emg_artifacts_filter/outputs/qc/raw_base_annotations.csv


In [4]:
# check
channels_table = pd.DataFrame(
    {
        "channel": raw_base.ch_names,
        "type": raw_base.get_channel_types(),
        "bad": [ch in raw_base.info["bads"] for ch in raw_base.ch_names],
    }
)

display(channels_table)
print(f"Sampling rate: {raw_base.info['sfreq']} Hz")
print(f"Duration: {duration_s:.2f} s")
print("Bad channels:", raw_base.info["bads"] or "none")

raw_base.plot()


,channel,type,bad
0,RF R,eeg,False
1,BF R,eeg,False
2,TA R,eeg,False
3,GM R,eeg,False
4,RF L,eeg,False
5,BF L,eeg,False
6,TA L,eeg,False
7,GM L,eeg,False


Sampling rate: 4000.0 Hz
Duration: 117.53 s
Bad channels: none


In [ ]:
# Сохранение всех аннотаций в файл

raw_base.annotations.save(
    annotations_path,
    overwrite=True,
)

print(f"Сохранено аннотаций: {len(raw_base.annotations)}")
print("Файл:", annotations_path)

In [5]:
emg_channels = ["GM R", "GM L", "RF R", "RF L", "TA R", "TA L", "BF R", "BF L"]
right_channels = ["GM R", "RF R", "TA R", "BF R"]
left_channels = ["GM L", "RF L", "TA L", "BF L"]

excluded_channels = ["TA L"]

print("EMG channels:", emg_channels)
print("Right:", right_channels)
print("Left:", left_channels)
print("Excluded:", excluded_channels)


EMG channels: ['GM R', 'GM L', 'RF R', 'RF L', 'TA R', 'TA L', 'BF R', 'BF L']
Right: ['GM R', 'RF R', 'TA R', 'BF R']
Left: ['GM L', 'RF L', 'TA L', 'BF L']
Excluded: ['TA L']


In [6]:
# Global CAR

car_channels = [ch for ch in emg_channels if ch not in excluded_channels]

raw_car_global = raw_base.copy()

avg = raw_car_global.get_data(picks=car_channels).mean(axis=0)
idx = [raw_car_global.ch_names.index(ch) for ch in car_channels]

raw_car_global._data[idx] -= avg

car_global_path = data_processed_dir / "recording_car_global_raw.fif"
raw_car_global.save(car_global_path, overwrite=True)

print("Saved:", car_global_path)
raw_car_global.plot()


Saved: /Users/user/emg_artifacts_filter/data/processed/recording_car_global_raw.fif


In [7]:
# CAR отдельно справа и слева

raw_car_left_right = raw_base.copy()

for channels in [right_channels, left_channels]:
    car_channels = [
        ch for ch in channels
        if ch not in excluded_channels
    ]

    avg = raw_car_left_right.get_data(picks=car_channels).mean(axis=0)
    idx = [raw_car_left_right.ch_names.index(ch) for ch in car_channels]

    raw_car_left_right._data[idx] -= avg

car_left_right_path = (
    data_processed_dir / "recording_car_left_right_raw.fif"
)
raw_car_left_right.save(car_left_right_path, overwrite=True)

print("Saved:", car_left_right_path)

raw_car_left_right.plot()


Saved: /Users/user/emg_artifacts_filter/data/processed/recording_car_left_right_raw.fif


In [ ]:
# GED: параметры и список каналов

GED_LABEL = "artifact"
GED_LOWPASS_HZ = 10.0  # Двухфазная волна медленная; острые EMG-пики не вычитаем.
GED_BASELINE_S = 0.5  # Чистая EMG до и после каждого артефакта.
GED_GUARD_S = 0.05  # Не берём хвост артефакта в baseline.
GED_REGULARIZATION = 0.05  # Стабилизирует ковариацию normal EMG.
GED_MIN_POWER_RATIO = 3.0  # Берём компоненты заметно сильнее в artifact.
GED_FADE_S = 0.025  # Плавные края вычитания без скачков.

ged_channels = [
    ch for ch in emg_channels
    if ch not in excluded_channels
]

ged_model_path = qc_dir / "ged_artifact_model_and_masks.npz"
ged_output_path = data_processed_dir / "recording_ged_masked_raw.fif"

missing_channels = [
    ch for ch in ged_channels
    if ch not in raw_base.ch_names
]
if missing_channels:
    raise ValueError(f"В записи отсутствуют каналы: {missing_channels}")

print("GED channels:", ged_channels)
print("Excluded from GED:", excluded_channels)
print("Low-pass for artifact model, Hz:", GED_LOWPASS_HZ)


## GED-компонента артефакта с локальной normal EMG

GED ищет комбинацию каналов, мощную внутри `artifact`, но слабую в соседних чистых участках: $C_{artifact}u = \lambda C_{normal}u$. Модель строится по копии ниже 10 Гц и вычитает только медленную компоненту, поэтому узкие EMG-пики напрямую не проецируются.

In [ ]:
# 1. Маски артефактов и локальной normal EMG
artifact_mask = np.zeros(raw_base.n_times, dtype=bool)
normal_mask = np.zeros(raw_base.n_times, dtype=bool)
artifact_intervals = []

for annotation in raw_base.annotations:
    if GED_LABEL.lower() not in annotation["description"].lower():
        continue

    onset_from_start = annotation["onset"] - raw_base.first_time
    start = int(raw_base.time_as_index(onset_from_start, use_rounding=True)[0])
    stop = int(raw_base.time_as_index(
        onset_from_start + annotation["duration"], use_rounding=True
    )[0])
    start = max(0, start)
    stop = min(raw_base.n_times, max(start + 1, stop))

    artifact_mask[start:stop] = True
    artifact_intervals.append((start, stop))

if not artifact_mask.any():
    raise ValueError(f"Нет аннотаций с меткой {GED_LABEL!r}")

baseline = int(round(GED_BASELINE_S * sfreq))
guard = int(round(GED_GUARD_S * sfreq))
for start, stop in artifact_intervals:
    normal_mask[max(0, start - guard - baseline):max(0, start - guard)] = True
    normal_mask[min(raw_base.n_times, stop + guard):min(raw_base.n_times, stop + guard + baseline)] = True
normal_mask &= ~artifact_mask

print("Artifact intervals:", len(artifact_intervals))
print("Artifact duration, s:", round(artifact_mask.sum() / sfreq, 3))
print("Local normal EMG duration, s:", round(normal_mask.sum() / sfreq, 3))


In [ ]:
# 2. GED по медленной части сигнала
X = raw_base.get_data(picks=ged_channels)
lowpass = signal.butter(
    4, GED_LOWPASS_HZ, btype="lowpass", fs=sfreq, output="sos"
)
X_slow = signal.sosfiltfilt(lowpass, X, axis=1)

def covariance(data):
    centered = data - data.mean(axis=1, keepdims=True)
    return centered @ centered.T / (centered.shape[1] - 1)

C_artifact = covariance(X_slow[:, artifact_mask])
C_normal = covariance(X_slow[:, normal_mask])
ridge = GED_REGULARIZATION * np.trace(C_normal) / len(ged_channels)
C_normal_reg = C_normal + ridge * np.eye(len(ged_channels))

eigenvalues, eigenvectors = linalg.eigh(C_artifact, C_normal_reg)
selected = eigenvalues > GED_MIN_POWER_RATIO
if not selected.any():
    raise ValueError("GED не нашёл компонент, специфичных для артефакта")
ged_filters = eigenvectors[:, selected]
components = ged_filters.T @ X_slow

# Регрессия возвращает каждую выбранную компоненту в пространство каналов.
components_artifact = components[:, artifact_mask]
spatial_patterns = (
    X_slow[:, artifact_mask] @ components_artifact.T
    @ np.linalg.pinv(components_artifact @ components_artifact.T)
)
artifact_model = spatial_patterns @ components

eigenvalue_table = pd.DataFrame({
    "artifact / normal power": eigenvalues[::-1],
    "remove": selected[::-1],
})
display(eigenvalue_table)
display(pd.DataFrame(
    spatial_patterns, index=ged_channels,
    columns=[f"component {i + 1}" for i in range(selected.sum())],
))


In [ ]:
# 3. Вычитаем только медленную GED-компоненту внутри разметки
apply_weight = np.zeros(raw_base.n_times)
fade = int(round(GED_FADE_S * sfreq))
for start, stop in artifact_intervals:
    length = stop - start
    edge = min(fade, length // 2)
    window = np.ones(length)
    if edge:
        ramp = 0.5 - 0.5 * np.cos(np.linspace(0, np.pi, edge))
        window[:edge] = ramp
        window[-edge:] = ramp[::-1]
    apply_weight[start:stop] = window

X_clean = X - artifact_model * apply_weight
raw_ged = raw_base.copy().load_data()
ged_indices = [raw_ged.ch_names.index(ch) for ch in ged_channels]
raw_ged._data[ged_indices] = X_clean

unchanged_outside = np.array_equal(
    raw_ged.get_data(picks=ged_channels)[:, ~artifact_mask],
    X[:, ~artifact_mask],
)
print("Outside annotations unchanged:", unchanged_outside)


In [ ]:
# 4. Проверка: медленная волна уменьшается, быстрые пики сохраняются
records = {
    "original": raw_base,
    "global CAR": raw_car_global,
    "left/right CAR": raw_car_left_right,
    "GED slow component": raw_ged,
}

rms_table = pd.DataFrame({
    name: np.sqrt(np.mean(
        record.get_data(picks=ged_channels)[:, artifact_mask] ** 2, axis=1
    ))
    for name, record in records.items()
}, index=ged_channels)
rms_table.index.name = "channel"
display(rms_table)

X_clean_slow = signal.sosfiltfilt(lowpass, X_clean, axis=1)
X_fast = X - X_slow
X_clean_fast = X_clean - X_clean_slow
quality = pd.Series({
    "slow RMS: artifact / local normal, before": np.sqrt(np.mean(X_slow[:, artifact_mask] ** 2)) / np.sqrt(np.mean(X_slow[:, normal_mask] ** 2)),
    "slow RMS: artifact / local normal, after": np.sqrt(np.mean(X_clean_slow[:, artifact_mask] ** 2)) / np.sqrt(np.mean(X_slow[:, normal_mask] ** 2)),
    "fast RMS after / before": np.sqrt(np.mean(X_clean_fast[:, artifact_mask] ** 2)) / np.sqrt(np.mean(X_fast[:, artifact_mask] ** 2)),
}, name="value")
display(quality.to_frame())


In [ ]:
# 5. Визуальное сравнение первого артефакта с контекстом
start, stop = artifact_intervals[0]
plot_channel = ged_channels[0]
context = int(round(0.25 * sfreq))
view_start = max(0, start - context)
view_stop = min(raw_base.n_times, stop + context)
time = raw_base.times[view_start:view_stop]

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True, sharey=True)
for axis, (name, record) in zip(axes, records.items()):
    values = record.get_data(picks=[plot_channel])[0, view_start:view_stop]
    axis.plot(time, values, linewidth=0.9)
    axis.axvspan(raw_base.times[start], raw_base.times[stop - 1], color="tab:orange", alpha=0.15)
    axis.set_title(name)
    axis.set_ylabel("V")
axes[-1].set_xlabel("Time, s")
fig.suptitle(f"{plot_channel}: first artifact interval")
fig.tight_layout()
plt.show()


In [ ]:
# 6. Сохранение очищенной записи и параметров GED
raw_ged.save(ged_output_path, overwrite=True)

np.savez(
    ged_model_path,
    channels=np.asarray(ged_channels),
    covariance_artifact=C_artifact,
    covariance_normal=C_normal,
    eigenvalues=eigenvalues,
    ged_filters=ged_filters,
    spatial_patterns=spatial_patterns,
    artifact_mask=artifact_mask,
    normal_mask=normal_mask,
    lowpass_hz=GED_LOWPASS_HZ,
)

output_files = {
    "global CAR": car_global_path,
    "left/right CAR": car_left_right_path,
    "GED slow component": ged_output_path,
}
for name, path in output_files.items():
    print(f"{name}: {path}")
print("GED model and masks:", ged_model_path)
